In [1]:
# Check if unsloth is already installed (to avoid re-running if you just restarted the python kernel)
try:
    import unsloth
    print("Unsloth is already installed!")
except ImportError:
    # Fast-install the optimized Colab wheels
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
    !pip install --no-deps "xformers<0.0.29" "triton==3.0.0" trl peft --quiet
    print("Unsloth installation complete!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Clone the official Unsloth notebooks repository
!git clone https://github.com/unslothai/notebooks.git

# Move into the notebooks directory
%cd notebooks

Cloning into 'notebooks'...
remote: Enumerating objects: 26745, done.
remote: Counting objects: 100% (12934/12934), done.
remote: Compressing objects: 100% (1233/1233), done.
remote: Total 26745 (delta 12632), reused 11715 (delta 11701), pack-reused 13811 (from 2)
Receiving objects: 100% (26745/26745), 119.41 MiB | 28.08 MiB/s, done.
Resolving deltas: 100% (25088/25088), done.
/content/notebooks


In [3]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding versions!
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.0.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [5]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
import os

# 1. Apply Qwen's ChatML template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# 2. Format the conversational structure
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = []
    for convo in convos:
        formatted_text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        texts.append(formatted_text + EOS_TOKEN)
    return { "text" : texts }

drive_dataset_path = "/content/drive/MyDrive/input.jsonl"

if os.path.exists(drive_dataset_path):
    dataset = load_dataset("json", data_files=drive_dataset_path, split="train")
    dataset = dataset.map(formatting_prompts_func, batched=True)
    print("Dataset loaded successfully from Google Drive!")
else:
    raise FileNotFoundError(f"Could not find 'input.jsonl' at {drive_dataset_path}. Please make sure you uploaded it to your Google Drive root.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Dataset loaded successfully from Google Drive!


In [6]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Loads the base Qwen model and layers your saved adapter on top of it automatically!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/unsloth_outputs/final_lora_adapter", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Re-apply the Qwen conversational layout
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

FastLanguageModel.for_inference(model) # Enable 2x faster native inference
print("Model and adapters loaded successfully from Google Drive! Ready for inference.")

==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.0.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model and adapters loaded successfully from Google Drive! Ready for inference.


INFERENCE

In [9]:
# alpaca_prompt = Copied from above

FastLanguageModel.for_inference(model) # Enable native 2x faster inference
prompt = [{"role": "user", "content": "show me a hadamard operation being applied to a qubit state | 0 > + | 1 > "}]
inputs = tokenizer(
[
    formatting_prompts_func({
        "messages": [prompt]
    })["text"][0]
], return_tensors = "pt").to("cuda")

from transformers import TextIteratorStreamer
streamer = TextIteratorStreamer(tokenizer)

outputs = model.generate(**inputs,streamer=streamer, max_new_tokens = 500, use_cache = True)
unformatted_output=tokenizer.batch_decode(outputs)

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [10]:
print("Unformatted Output:\n", unformatted_output[0])

Unformatted Output:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
show me a hadamard operation being applied to a qubit state | 0 > + | 1 > <|im_end|>
<|im_end|>
<|im_start|>assistant
To visualize this conceptually, we can use the Bloch sphere representation:

```python
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_hadamard():
    fig = plt.figure(figsize=(8, 4))
    
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.set_title('Initial State: |0> + |1>')
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x = np.outer(np.sin(v), np.cos(u))
    y = np.outer(np.sin(v), np.sin(u))
    z = np.outer(np.cos(v), np.ones_like(u))
    ax1.plot_surface(x, y, z, color='lightblue')

    ax2 = fig.add_subplot(122, projection='3d')
    ax2.set_title('After Hadamard Gate')
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)


taking out python script and formatting it into code from json